In [ ]:
import io, os                                   # to explore directories
import pandas as pd                             # pandas
import paramiko                                 # to connect to the server
from dotenv import load_dotenv                  #to read secrets from the .env file

load_dotenv()
SSH_HOST = os.environ["SSH_HOST"]
SSH_USER = os.environ["SSH_USER"]
REMOTE_ROSTER_DIR = os.environ["REMOTE_ROSTER_DIR"]

Get the server password

In [3]:
import ipywidgets as widgets
from IPython.display import display

password_box = widgets.Password(description="Password:")
display(password_box)

Password(description='Password:')

Connect to the server

In [4]:
password = password_box.value

client = paramiko.SSHClient()
client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
client.connect(SSH_HOST, port=int(os.environ.get("SSH_PORT", 22)), username=SSH_USER, password=password)

sftp = client.open_sftp()
print("Connected")

Connected


In [5]:
transfer_out_df = pd.DataFrame()
separation_df = pd.DataFrame()

In [14]:
folder_2025 = f"{REMOTE_ROSTER_DIR}2025"
folder_months = sftp.listdir(folder_2025)
folder_months

['January',
 'August',
 'February',
 'June',
 'July',
 'May',
 'April',
 'March',
 'September',
 'October',
 'November',
 'December']

In [ ]:
test_file = "HX_I254_DATA_12222025_050000.xlsx"
file_path = f"/var/www/html/CWANU/JurisFiles/2025/December/12222025/{test_file}"

with sftp.open(file_path, 'rb') as f:
    data = f.read()
    df = 

df.head()

In [28]:
cols_need = ['Type Change Description/Action Description', 'Employee (EE) ID', 'Employee Name', 'Bargaining Unit', 'Location/Business Unit',
       'Job Indicator', 'Jobcode', 'Job Title Abbr Name', 'Annual Pay Rate 1.0 FTE', 'Step', 'Appt. %',
       'Employee Class Description', 'Appt Rep Code','Most Recent Date of Hire in Empl Record','Department ID', 'Department ID Description']

In [30]:
def filtering_df(file_dir, file_name, cols):
    with sftp.open(file_dir, "rb") as f:
        data = f.read()
        df = pd.read_excel(io.BytesIO(data))
        df = df[cols]
        df['Juris_file'] = file_name
        df_transfer_out = df[df['actionDescription'] == 'TRANSFER OUT']          #filter transfer out rows
        df_separation = df[df['actionDescription'] == 'SEPARATION']              # filter separation rows
    return df_transfer_out, df_separation
        



In [ ]:
for month in folder_months:
    month_dir = f"{folder_2025}/{month}"
    week_folders = sftp.listdir(month_dir)
    for week in week_folders:
        if week == ".DS_Store":                 #skip the hidden .DS_Store folders
            continue
        week_dir = f"{month_dir}/{week}"
        print(week_dir)
        files = sftp.listdir(week_dir)
        prefixes = ('HX', 'RX', 'TX')
        for file in files:
            if file.startswith(prefixes):
                print(f"for {month}'s, {week} juris files are {file}")
    